<a href="https://colab.research.google.com/github/hollyemblem/out-of-domain-mmd/blob/mmd-experiments/domain_shift_experiments_calibration_BTC_SEC_WNUT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MMD Experiments

### Method:

"Intuitively, the MMD test evaluates whether there is a significant difference between two distributions: a higher MMD value suggests a greater disparity between the distributions. The test essentially disproves the null hypothesis that the distributions are identical—when the MMD statistic is significantly high. In our study, the MMD values can appear negative due to estimation errors in smaller samples or due to the kernel choice affecting the calculation. However, the absolute value of MMD should be considered. Typically, a threshold for significance is set, above which the null hypothesis can be rejected. We use 0.05 as our threshold."

Source: https://link.springer.com/article/10.1007/s10579-024-09754-8#Sec3


### Dataset trials

- BTC and SEC calibration from paper https://link.springer.com/article/10.1007/s10579-024-09754-8#Sec3

- BTC and WNUT-17 calibration from the paper  https://link.springer.com/article/10.1007/s10579-024-09754-8#Sec3

Source: https://arxiv.org/pdf/2202.11176

### GPU Setup

In [8]:
import torch

torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
print(torch.cuda.is_available())

True


#### Installing Required Libraries


In [10]:
# @title
#!pip install -U sentence-transformers
#https://huggingface.co/efederici/sentence-bert-base



In [11]:
!pip install pytorch-ignite

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:

%cd /content/drive/My Drive/Colab Notebooks/domain_shift_experiments_05_2026

/content/drive/My Drive/Colab Notebooks/domain_shift_experiments_05_2026


In [14]:
!pip install datasets==3.6.0

## Import Libraries

In [15]:
from google.colab import userdata
import pandas as pd
import os
import requests
import numpy as np
from sentence_transformers import SentenceTransformer
from ignite.metrics import MaximumMeanDiscrepancy
import random

In [135]:
sample_size = 948

## Broad Twitter Corpus

In [136]:
from datasets import load_dataset
#https://huggingface.co/datasets/GateNLP/broad_twitter_corpus?library=datasets
ds = load_dataset("GateNLP/broad_twitter_corpus")

In [137]:
twitter_corpus = ds["train"].to_pandas()

In [138]:
twitter_corpus["tokens"] = twitter_corpus["tokens"].apply(lambda x: " ".join(x))

In [139]:
twitter_corpus

,id,tokens,ner_tags
0,0,"I hate the words chunder , vomit and puke . BU...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,1,♥ . . ) ) ( ♫ . ( ړײ ) ♫ . ♥ . « ▓ » ♥ . ♫ . ....,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,2,Alesan kenapa mlm kita lbh srg galau Poconggg ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,3,Complete Tosca on the tube http://t.co/O90deSLB,"[0, 0, 0, 0, 0, 0]"
4,4,Think you call that smash and grab . # Gateshe...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, ..."
...,...,...,...
5337,5337,Watch : Attorney General Eric Holder 's NAACP ...,"[0, 0, 0, 0, 1, 2, 0, 3, 0, 0]"
5338,5338,Filibuster fight gears up in Senate . Watch li...,"[0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5339,5339,Watch live : Senator Gillibrand newser on sexu...,"[0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0]"
5340,5340,"Watch Live : President Obama , George H . W . ...","[0, 0, 0, 1, 2, 0, 1, 2, 2, 2, 2, 2, 0, 0, 0, ..."


In [140]:
twitter_corpus_list = twitter_corpus['tokens'].to_list()

In [141]:
twitter_corpus_list_sample = random.sample(twitter_corpus_list, sample_size)

### SEC Dataset
https://huggingface.co/datasets/tner/fin?library=datasets

In [142]:
sec = load_dataset("tner/fin")

In [143]:
sec_training = sec['train'].to_pandas()

In [144]:
sec_training["tokens"] = sec_training["tokens"].apply(lambda x: " ".join(x))

In [145]:
len(sec_training_list)

1018

In [146]:
sec_training_list = sec_training['tokens'].to_list()

In [147]:
import random
sec_training_sample = random.sample(sec_training_list, sample_size)

## WNUT-17
Source: https://huggingface.co/datasets/leondz/wnut_17

In [148]:
wnut  = load_dataset("leondz/wnut_17")

In [149]:
wnut_train = wnut['train'].to_pandas()

In [150]:
wnut_train["tokens"] = wnut_train["tokens"].apply(lambda x: " ".join(x))

In [151]:
wnut_train_list = wnut_train['tokens'].to_list()

In [152]:
wnut_training_sample = random.sample(wnut_train_list, sample_size)

### Sentence Embeddings Setup

In [153]:
print(torch.cuda.get_device_name(0))

model = SentenceTransformer(
    'efederici/sentence-bert-base',
    device='cuda'
)

print(model.device)

NVIDIA A100-SXM4-80GB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

cuda:0


In [154]:
##train embeddings
model = model.to(device)

In [155]:

twitter_embeddings = model.encode(
    twitter_corpus_list_sample,
    batch_size=sample_size,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(twitter_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(948, 768)


In [156]:
##sec embeddings

sec_embeddings = model.encode(
    sec_training_sample,
    batch_size=sample_size,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(sec_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(948, 768)


In [157]:
##val embeddings

wnut_embeddings = model.encode(
    wnut_training_sample,
    batch_size=sample_size,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(wnut_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(948, 768)


In [158]:
##Maximum Mean Discrepancy https://docs.pytorch.org/ignite/generated/ignite.metrics.MaximumMeanDiscrepancy.html

In [159]:
#Example code from https://github.com/emanuele/kernel_two_sample_test/blob/master/kernel_two_sample_test.py
from __future__ import division
import numpy as np
from sys import stdout
from sklearn.metrics import pairwise_kernels


In [160]:
#From: https://github.com/emanuele/kernel_two_sample_test/blob/master/kernel_two_sample_test.py
def MMD2u(K, m, n):
    """The MMD^2_u unbiased statistic.
    """
    Kx = K[:m, :m]
    Ky = K[m:, m:]
    Kxy = K[:m, m:]
    return 1.0 / (m * (m - 1.0)) * (Kx.sum() - Kx.diagonal().sum()) + \
        1.0 / (n * (n - 1.0)) * (Ky.sum() - Ky.diagonal().sum()) - \
        2.0 / (m * n) * Kxy.sum()


def compute_null_distribution(K, m, n, iterations=10000, verbose=False,
                              random_state=None, marker_interval=1000):
    """Compute the bootstrap null-distribution of MMD2u.
    """
    if type(random_state) == type(np.random.RandomState()):
        rng = random_state
    else:
        rng = np.random.RandomState(random_state)

    mmd2u_null = np.zeros(iterations)
    for i in range(iterations):
        if verbose and (i % marker_interval) == 0:
            print(i),
            stdout.flush()
        idx = rng.permutation(m+n)
        K_i = K[idx, idx[:, None]]
        mmd2u_null[i] = MMD2u(K_i, m, n)

    if verbose:
        print("")

    return mmd2u_null


def compute_null_distribution_given_permutations(K, m, n, permutation,
                                                 iterations=None):
    """Compute the bootstrap null-distribution of MMD2u given
    predefined permutations.

    Note:: verbosity is removed to improve speed.
    """
    if iterations is None:
        iterations = len(permutation)

    mmd2u_null = np.zeros(iterations)
    for i in range(iterations):
        idx = permutation[i]
        K_i = K[idx, idx[:, None]]
        mmd2u_null[i] = MMD2u(K_i, m, n)

    return mmd2u_null


def kernel_two_sample_test(X, Y, kernel_function='rbf', iterations=10000,
                           verbose=False, random_state=None, **kwargs):
    """Compute MMD^2_u, its null distribution and the p-value of the
    kernel two-sample test.

    Note that extra parameters captured by **kwargs will be passed to
    pairwise_kernels() as kernel parameters. E.g. if
    kernel_two_sample_test(..., kernel_function='rbf', gamma=0.1),
    then this will result in getting the kernel through
    kernel_function(metric='rbf', gamma=0.1).
    """
    m = len(X)
    n = len(Y)
    XY = np.vstack([X, Y])
    K = pairwise_kernels(XY, metric=kernel_function, **kwargs)
    mmd2u = MMD2u(K, m, n)
    if verbose:
        print("MMD^2_u = %s" % mmd2u)
        print("Computing the null distribution.")

    mmd2u_null = compute_null_distribution(K, m, n, iterations,
                                           verbose=verbose,
                                           random_state=random_state)
    p_value = max(1.0/iterations, (mmd2u_null > mmd2u).sum() /
                  float(iterations))
    if verbose:
        print("p-value ~= %s \t (resolution : %s)" % (p_value, 1.0/iterations))

    return mmd2u, mmd2u_null, p_value

## Twitter and WNUT-17 Calibration

In [161]:
# Using https://github.com/emanuele/kernel_two_sample_test/blob/master/kernel_two_sample_test.py to calculate mmu^2 unbiased as per paper
#MMD usage paper = https://link.springer.com/article/10.1007/s10579-024-09754-8#Sec3
from scipy.spatial.distance import pdist
from sklearn.metrics.pairwise import pairwise_kernels
x = torch.tensor(twitter_embeddings, dtype=torch.float32, device=device).detach().cpu().numpy()
y = torch.tensor(wnut_embeddings, dtype=torch.float32, device=device).detach().cpu().numpy()

m = len(x)
n = len(y)

# Pool the distributions
Z = np.vstack([x, y])

# sigma = Calculate the median Euclidean distance between points
sigma = np.median(
    pdist(Z, metric="euclidean") #Review paper here: https://www.jmlr.org/papers/volume13/gretton12a/gretton12a.pdf
)

print("sigma:", sigma)

# sklearn RBF:
# exp(-gamma * ||x-y||²)
#
# Paper:
# exp(-(1/sigma) * ||x-y||²)
#
# Therefore gamma = 1/sigma

K = pairwise_kernels(
    Z,
    metric="rbf",
    gamma=1.0 / sigma
)

mmd2_u = MMD2u(K, m, n)

print("Unbiased MMD²:", mmd2_u)

sigma: 1.069090142296261
Unbiased MMD²: 0.028099298


In [162]:
x = torch.tensor(twitter_embeddings, dtype=torch.float32, device=device)
y = torch.tensor(wnut_embeddings, dtype=torch.float32, device=device)

n = min(len(x), len(y))
batch_size = sample_size

mmd = MaximumMeanDiscrepancy(var=1.0, device=device)
mmd.reset()

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)

    xb = x[start:end]
    yb = y[start:end]

    if len(xb) == len(yb):
        mmd.update((xb, yb))

score = mmd.compute()
print(score)

0.1521805077791214


## Twitter and SEC Calibration

In [163]:
# Using https://github.com/emanuele/kernel_two_sample_test/blob/master/kernel_two_sample_test.py to calculate mmu^2 unbiased as per paper
#MMD usage paper = https://link.springer.com/article/10.1007/s10579-024-09754-8#Sec3
from scipy.spatial.distance import pdist
from sklearn.metrics.pairwise import pairwise_kernels
x = torch.tensor(twitter_embeddings, dtype=torch.float32, device=device).detach().cpu().numpy()
y = torch.tensor(sec_embeddings, dtype=torch.float32, device=device).detach().cpu().numpy()

m = len(x)
n = len(y)

# Pool the distributions
Z = np.vstack([x, y])

# sigma = Calculate the median Euclidean distance between points
sigma = np.median(
    pdist(Z, metric="euclidean") #Review paper here: https://www.jmlr.org/papers/volume13/gretton12a/gretton12a.pdf
)

print("sigma:", sigma)

# sklearn RBF:
# exp(-gamma * ||x-y||²)
#
# Paper:
# exp(-(1/sigma) * ||x-y||²)
#
# Therefore gamma = 1/sigma

K = pairwise_kernels(
    Z,
    metric="rbf",
    gamma=1.0 / sigma
)

mmd2_u = MMD2u(K, m, n)

print("Unbiased MMD²:", mmd2_u)

sigma: 1.1313836547001381
Unbiased MMD²: 0.13896132


In [164]:
#kernel_two_sample_test(x,y,kernel_function="rbf",
 #   gamma=1.0 / sigma,
  #  iterations=1000,
   # random_state=1,
   # verbose=True)

In [165]:
x = torch.tensor(twitter_embeddings, dtype=torch.float32, device=device)
y = torch.tensor(sec_embeddings, dtype=torch.float32, device=device)

n = min(len(x), len(y))
batch_size = sample_size

mmd = MaximumMeanDiscrepancy(var=1.0, device=device)
mmd.reset()

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)

    xb = x[start:end]
    yb = y[start:end]

    if len(xb) == len(yb):
        mmd.update((xb, yb))

score = mmd.compute()
print(score)

0.34300705790519714


## WNUT-17 and SEC

In [168]:
# Using https://github.com/emanuele/kernel_two_sample_test/blob/master/kernel_two_sample_test.py to calculate mmu^2 unbiased as per paper
#MMD usage paper = https://link.springer.com/article/10.1007/s10579-024-09754-8#Sec3
from scipy.spatial.distance import pdist
from sklearn.metrics.pairwise import pairwise_kernels
x = torch.tensor(wnut_embeddings, dtype=torch.float32, device=device).detach().cpu().numpy()
y = torch.tensor(sec_embeddings, dtype=torch.float32, device=device).detach().cpu().numpy()

m = len(x)
n = len(y)

# Pool the distributions
Z = np.vstack([x, y])

# sigma = Calculate the median Euclidean distance between points
sigma = np.median(
    pdist(Z, metric="euclidean") #Review paper here: https://www.jmlr.org/papers/volume13/gretton12a/gretton12a.pdf
)

print("sigma:", sigma)

# sklearn RBF:
# exp(-gamma * ||x-y||²)
#
# Paper:
# exp(-(1/sigma) * ||x-y||²)
#
# Therefore gamma = 1/sigma

K = pairwise_kernels(
    Z,
    metric="rbf",
    gamma=1.0 / sigma
)

mmd2_u = MMD2u(K, m, n)

print("Unbiased MMD²:", mmd2_u)

sigma: 1.1214517034991909
Unbiased MMD²: 0.18207097


In [170]:
x = torch.tensor(wnut_embeddings, dtype=torch.float32, device=device)
y = torch.tensor(sec_embeddings, dtype=torch.float32, device=device)

n = min(len(x), len(y))
batch_size = sample_size

mmd = MaximumMeanDiscrepancy(var=1.0, device=device)
mmd.reset()

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)

    xb = x[start:end]
    yb = y[start:end]

    if len(xb) == len(yb):
        mmd.update((xb, yb))

score = mmd.compute()
print(score)

0.393655389547348
